# SAMPLE PRE-PROCESSING

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm, AnovaRM
from statsmodels.multivariate.manova import MANOVA

In [2]:
df = pd.read_csv("/content/autism.csv")

In [3]:
data = df.rename(columns={
    "Student ID": "ID",
    "Chronological Age": "CA",
    "Developmental Age": "DA",
    "Baseline MAND": "MAND0",
    "Baseline TACT": "TACT0",
    "Baseline INTRA": "INTRA0",
    "Support Level": "Support",
    "Intervention Approach": "Approach",
    "Intervention Intensity": "Intensity",
})

data = data.drop_duplicates().dropna().reset_index(drop=True)

support_order = ["Level 1", "Level 2", "Level 3"]
data["Support"] = pd.Categorical(data["Support"], categories=support_order, ordered=True)

alpha = 0.05
power = 0.80
rng = np.random.default_rng(42)

resample_data = data.sample(n=2000, random_state=42).reset_index(drop=True)
distance_data = data.sample(n=300, random_state=42).reset_index(drop=True)
n_permutation = 1000

print(df.shape, "->", data.shape)
print(data.dtypes.head(9))

(20045, 39) -> (19350, 39)
ID              int64
CA              int64
MAND0         float64
TACT0           int64
INTRA0        float64
DA              int64
Support      category
Approach       object
Intensity       int64
dtype: object


In [4]:
def interpret(value, thresholds, labels):
    return labels[int(np.searchsorted(thresholds, abs(value), side="right"))]

def decide(p_value):
    return "reject H0" if p_value < alpha else "fail to reject H0"


## SAMPLE SIZE

In [5]:
z_alpha = stats.norm.ppf(1 - alpha / 2)
z_beta = stats.norm.ppf(power)

group_1 = data.loc[data["Approach"] == "NET", "W10MAND"]
group_2 = data.loc[data["Approach"] == "DTT", "W10MAND"]
sigma = np.sqrt((group_1.var(ddof=1) + group_2.var(ddof=1)) / 2)
delta = abs(group_1.mean() - group_2.mean())

difference = data["W10MAND"] - data["W01MAND"]
sigma_difference = difference.std(ddof=1)
delta_paired = abs(difference.mean())

grand_mean = data["W10MAND"].mean()
between = sum(len(g) * (g["W10MAND"].mean() - grand_mean) ** 2
              for _, g in data.groupby("Support", observed=True)) / len(data)
cohen_f = np.sqrt(between / data["W10MAND"].var(ddof=1))

table = pd.crosstab(data["Approach"], data["Support"])
chi_square = stats.chi2_contingency(table).statistic
cohen_w = np.sqrt(chi_square / table.values.sum())

r = data["MAND0"].corr(data["INTRA0"])
fisher_z = 0.5 * np.log((1 + r) / (1 - r))

u = 3
v = 3

requirement = pd.DataFrame({
    "Test Family": [
        "Independent t & Wilcoxon Rank-Sum",
        "Paired t & Wilcoxon Signed-Rank",
        "One-Way ANOVA, RM ANOVA, Kruskal-Wallis, Friedman",
        "Two-Way ANOVA, SRH, ART, Chi-Square",
        "MANOVA, PERMANOVA, Wilks Lambda",
        "Pearson & Spearman Correlation",
    ],
    "Effect Size": [delta, delta_paired, cohen_f, cohen_w, cohen_f, r],
    "Required n": [
        2 * ((z_alpha + z_beta) * sigma / delta) ** 2,
        ((z_alpha + z_beta) * sigma_difference / delta_paired) ** 2,
        (z_alpha + z_beta) ** 2 / cohen_f ** 2,
        (z_alpha + z_beta) ** 2 / cohen_w ** 2,
        (z_alpha + z_beta) ** 2 * (u + v + 1) / (u * cohen_f ** 2),
        (z_alpha + z_beta) ** 2 / fisher_z ** 2,
    ],
})
requirement["Required n"] = np.ceil(requirement["Required n"])
requirement["Available n"] = len(data)
requirement["Sufficient"] = requirement["Available n"] >= requirement["Required n"]

print("Z(alpha):", round(z_alpha, 4), "| Z(beta):", round(z_beta, 4))
print(requirement.round(4).to_string(index=False))

Z(alpha): 1.96 | Z(beta): 0.8416
                                      Test Family  Effect Size  Required n  Available n  Sufficient
                Independent t & Wilcoxon Rank-Sum      22.2796        66.0        19350        True
                  Paired t & Wilcoxon Signed-Rank      24.5966         3.0        19350        True
One-Way ANOVA, RM ANOVA, Kruskal-Wallis, Friedman       0.6687        18.0        19350        True
              Two-Way ANOVA, SRH, ART, Chi-Square       0.4533        39.0        19350        True
                  MANOVA, PERMANOVA, Wilks Lambda       0.6687        41.0        19350        True
                   Pearson & Spearman Correlation       0.9272         3.0        19350        True


## SAMPLE DISTRIBUTION

In [6]:
deviation_bands = [(0.95, "none"), (0.90, "slight"), (0.80, "moderate"), (0.00, "strong")]

normality = []
for column in ["MAND0", "TACT0", "INTRA0", "W10MAND"]:
    values = data[column].sample(n=5000, random_state=42)
    w_stat, p_value = stats.shapiro(values)
    normality.append({
        "Variable": column,
        "W": round(w_stat, 4),
        "p-value": p_value,
        "Deviation": next(label for edge, label in deviation_bands if w_stat >= edge),
        "Decision": decide(p_value),
        "Route": "parametric" if p_value >= alpha else "non-parametric",
    })

print(pd.DataFrame(normality).to_string(index=False))

Variable      W      p-value Deviation  Decision          Route
   MAND0 0.9202 1.703768e-45    slight reject H0 non-parametric
   TACT0 0.9171 4.108010e-46    slight reject H0 non-parametric
  INTRA0 0.7802 2.185318e-63    strong reject H0 non-parametric
 W10MAND 0.9328 9.180756e-43    slight reject H0 non-parametric


# CONFIRMATORY DATA ANALYSIS

## INTER-GROUP ANALYSIS

### Parametric Tests For Continuous Numerical Inter-Group

#### Independent t-Test

In [7]:
group_1 = data.loc[data["Approach"] == "NET", "W10MAND"]
group_2 = data.loc[data["Approach"] == "DTT", "W10MAND"]

t_stat, p_value = stats.ttest_ind(group_1, group_2, equal_var=False)
degrees = len(group_1) + len(group_2) - 2
p_manual = 2 * (1 - stats.t.cdf(abs(t_stat), degrees))

print("NET:", len(group_1), "| mean:", round(group_1.mean(), 3))
print("DTT:", len(group_2), "| mean:", round(group_2.mean(), 3))
print("t:", round(t_stat, 4), "| df:", degrees)
print("Difference:", interpret(t_stat, [1, 2, 3, 5],
                               ["no", "weak", "moderate", "strong", "very strong"]))
print("p-value (two-tailed):", p_value)
print("p-value (CDF):", p_manual)
print("Decision:", decide(p_value))

NET: 6747 | mean: 98.324
DTT: 7392 | mean: 76.045
t: 29.0117 | df: 14137
Difference: very strong
p-value (two-tailed): 1.2096623980536697e-179
p-value (CDF): 0.0
Decision: reject H0


#### Paired t-Test

In [8]:
before = data["W01MAND"]
after = data["W10MAND"]

t_stat, p_value = stats.ttest_rel(after, before)
degrees = len(before) - 1
p_manual = 2 * (1 - stats.t.cdf(abs(t_stat), degrees))

print("Pairs:", len(before))
print("Mean before:", round(before.mean(), 3), "| Mean after:", round(after.mean(), 3))
print("Mean difference:", round((after - before).mean(), 3))
print("t:", round(t_stat, 4), "| df:", degrees)
print("Difference:", interpret(t_stat, [1, 2, 3, 5],
                               ["no", "weak", "moderate", "strong", "very strong"]))
print("p-value (two-tailed):", p_value)
print("p-value (CDF):", p_manual)
print("Decision:", decide(p_value))

Pairs: 19350
Mean before: 55.765 | Mean after: 80.362
Mean difference: 24.597
t: 258.2554 | df: 19349
Difference: very strong
p-value (two-tailed): 0.0
p-value (CDF): 0.0
Decision: reject H0


#### One-Way ANOVA Test

In [9]:
model = ols("W10MAND ~ C(Support)", data=data).fit()
table = anova_lm(model, typ=2)
print(table)

f_stat = table.loc["C(Support)", "F"]
df_effect = table.loc["C(Support)", "df"]
df_residual = table.loc["Residual", "df"]
p_value = 1 - stats.f.cdf(f_stat, df_effect, df_residual)

print(data.groupby("Support", observed=True)["W10MAND"].mean().round(3))
print("F:", round(f_stat, 4), "| df:", int(df_effect), int(df_residual))
print("p-value (CDF):", p_value)
print("Decision:", decide(p_value))

                  sum_sq       df            F  PR(>F)
C(Support)  1.851024e+07      2.0  7824.048224     0.0
Residual    2.288570e+07  19347.0          NaN     NaN
Support
Level 1    117.562
Level 2     73.649
Level 3     41.666
Name: W10MAND, dtype: float64
F: 7824.0482 | df: 2 19347
p-value (CDF): 1.1102230246251565e-16
Decision: reject H0


#### Two-Way ANOVA Test

In [10]:
model = ols("W10MAND ~ C(Support) * C(Approach)", data=data).fit()
table = anova_lm(model, typ=2)
print(table)

df_residual = table.loc["Residual", "df"]
for effect in ["C(Support)", "C(Approach)", "C(Support):C(Approach)"]:
    f_stat = table.loc[effect, "F"]
    df_effect = table.loc[effect, "df"]
    p_value = 1 - stats.f.cdf(f_stat, df_effect, df_residual)
    print(effect, "| F:", round(f_stat, 4), "| df:", int(df_effect), int(df_residual),
          "| p-value:", p_value, "|", decide(p_value))

                              sum_sq       df            F        PR(>F)
C(Support)              1.492767e+07      2.0  6391.221632  0.000000e+00
C(Approach)             2.618372e+05      2.0   112.104542  3.923291e-49
C(Support):C(Approach)  3.694362e+04      4.0     7.908633  2.297993e-06
Residual                2.258692e+07  19341.0          NaN           NaN
C(Support) | F: 6391.2216 | df: 2 19341 | p-value: 1.1102230246251565e-16 | reject H0
C(Approach) | F: 112.1045 | df: 2 19341 | p-value: 1.1102230246251565e-16 | reject H0
C(Support):C(Approach) | F: 7.9086 | df: 4 19341 | p-value: 2.297992639066493e-06 | reject H0


#### One-Way Repeated Measures ANOVA Test

In [11]:
subjects = data.sample(n=300, random_state=42)
long = subjects.melt(id_vars="ID", value_vars=["W01MAND", "W05MAND", "W10MAND"],
                     var_name="Week", value_name="MAND")

result = AnovaRM(long, depvar="MAND", subject="ID", within=["Week"]).fit()
print(result)

table = result.anova_table
f_stat = table.loc["Week", "F Value"]
p_value = 1 - stats.f.cdf(f_stat, table.loc["Week", "Num DF"], table.loc["Week", "Den DF"])

print(long.groupby("Week")["MAND"].mean().round(3))
print("F:", round(f_stat, 4), "| df:", table.loc["Week", "Num DF"], table.loc["Week", "Den DF"])
print("p-value (CDF):", p_value)
print("Decision:", decide(p_value))

                Anova
      F Value  Num DF  Den DF  Pr > F
-------------------------------------
Week 1017.3620 2.0000 598.0000 0.0000

Week
W01MAND    59.943
W05MAND    71.213
W10MAND    85.213
Name: MAND, dtype: float64
F: 1017.362 | df: 2.0 598.0
p-value (CDF): 1.1102230246251565e-16
Decision: reject H0


#### Two-Way Repeated Measures ANOVA Test

In [12]:
subjects = data.sample(n=300, random_state=42)
frames = []
for week in ["W01", "W05", "W10"]:
    for measure in ["MAND", "TACT"]:
        frames.append(pd.DataFrame({
            "ID": subjects["ID"].values,
            "Week": week,
            "Measure": measure,
            "Score": subjects[week + measure].values,
        }))
long = pd.concat(frames, ignore_index=True)

result = AnovaRM(long, depvar="Score", subject="ID", within=["Week", "Measure"]).fit()
print(result)

table = result.anova_table
for effect in table.index:
    f_stat = table.loc[effect, "F Value"]
    p_value = 1 - stats.f.cdf(f_stat, table.loc[effect, "Num DF"], table.loc[effect, "Den DF"])
    print(effect, "| F:", round(f_stat, 4), "| p-value:", p_value, "|", decide(p_value))

                    Anova
              F Value  Num DF  Den DF  Pr > F
---------------------------------------------
Week         1158.7687 2.0000 598.0000 0.0000
Measure       217.3540 1.0000 299.0000 0.0000
Week:Measure   13.9416 2.0000 598.0000 0.0000

Week | F: 1158.7687 | p-value: 1.1102230246251565e-16 | reject H0
Measure | F: 217.354 | p-value: 1.1102230246251565e-16 | reject H0
Week:Measure | F: 13.9416 | p-value: 1.208277498054855e-06 | reject H0


#### One-Way MANOVA Test

In [13]:
model = MANOVA.from_formula("W10MAND + W10TACT + W10INTRA ~ C(Support)", data=data)
result = model.mv_test()
print(result)

table = result.results["C(Support)"]["stat"]
wilks = table.loc["Wilks' lambda", "Value"]
f_stat = table.loc["Wilks' lambda", "F Value"]
p_value = table.loc["Wilks' lambda", "Pr > F"]

print("Wilks Lambda:", round(wilks, 6))
print("F:", round(f_stat, 4), "| df:", table.loc["Wilks' lambda", "Num DF"],
      table.loc["Wilks' lambda", "Den DF"])
print("p-value:", p_value)
print("Decision:", decide(p_value))

                    Multivariate linear model
                                                                  
------------------------------------------------------------------
       Intercept        Value  Num DF   Den DF    F Value   Pr > F
------------------------------------------------------------------
          Wilks' lambda 0.1618 3.0000 19345.0000 33395.6250 0.0000
         Pillai's trace 0.8382 3.0000 19345.0000 33395.6250 0.0000
 Hotelling-Lawley trace 5.1790 3.0000 19345.0000 33395.6250 0.0000
    Roy's greatest root 5.1790 3.0000 19345.0000 33395.6250 0.0000
------------------------------------------------------------------
                                                                  
------------------------------------------------------------------
        C(Support)       Value  Num DF   Den DF    F Value  Pr > F
------------------------------------------------------------------
           Wilks' lambda 0.5276 6.0000 38690.0000 2429.4977 0.0000
          Pillai

#### Two-Way MANOVA Test

In [14]:
model = MANOVA.from_formula("W10MAND + W10TACT + W10INTRA ~ C(Support) * C(Approach)",
                            data=data)
result = model.mv_test()

for effect in ["C(Support)", "C(Approach)", "C(Support):C(Approach)"]:
    table = result.results[effect]["stat"]
    wilks = table.loc["Wilks' lambda", "Value"]
    f_stat = table.loc["Wilks' lambda", "F Value"]
    p_value = table.loc["Wilks' lambda", "Pr > F"]
    print(effect, "| Wilks Lambda:", round(wilks, 6), "| F:", round(f_stat, 4),
          "| p-value:", p_value, "|", decide(p_value))

C(Support) | Wilks Lambda: 0.770865 | F: 895.8237 | p-value: 0.0 | reject H0
C(Approach) | Wilks Lambda: 0.746583 | F: 1014.264 | p-value: 0.0 | reject H0
C(Support):C(Approach) | Wilks Lambda: 0.9646 | F: 58.4821 | p-value: 1.855466014449271e-141 | reject H0


#### One-Way Repeated Measures MANOVA Test

In [15]:
subjects = data.sample(n=1000, random_state=42)
differences = pd.DataFrame({
    "dMAND": subjects["W10MAND"].values - subjects["W01MAND"].values,
    "dTACT": subjects["W10TACT"].values - subjects["W01TACT"].values,
    "dINTRA": subjects["W10INTRA"].values - subjects["W01INTRA"].values,
})

n = len(differences)
p = differences.shape[1]
mean_vector = differences.mean().values
covariance = np.cov(differences.values, rowvar=False)
t_squared = n * mean_vector @ np.linalg.inv(covariance) @ mean_vector
f_stat = t_squared * (n - p) / (p * (n - 1))
p_value = 1 - stats.f.cdf(f_stat, p, n - p)
wilks = 1 / (1 + t_squared / (n - 1))

print(differences.mean().round(3))
print("Hotelling T-squared:", round(t_squared, 4))
print("Wilks Lambda:", round(wilks, 6))
print("F:", round(f_stat, 4), "| df:", p, n - p)
print("p-value (CDF):", p_value)
print("Decision:", decide(p_value))

dMAND     24.592
dTACT     27.577
dINTRA    12.174
dtype: float64
Hotelling T-squared: 4064.8609
Wilks Lambda: 0.19728
F: 1352.241 | df: 3 997
p-value (CDF): 1.1102230246251565e-16
Decision: reject H0


#### Two-Way Repeated Measures MANOVA Test

In [16]:
subjects = data.sample(n=1000, random_state=42).reset_index(drop=True)
paired = pd.DataFrame({
    "dMAND": subjects["W10MAND"] - subjects["W01MAND"],
    "dTACT": subjects["W10TACT"] - subjects["W01TACT"],
    "dINTRA": subjects["W10INTRA"] - subjects["W01INTRA"],
    "Approach": subjects["Approach"],
    "Support": subjects["Support"],
})

model = MANOVA.from_formula("dMAND + dTACT + dINTRA ~ C(Approach) * C(Support)", data=paired)
result = model.mv_test()

for effect in ["C(Approach)", "C(Support)", "C(Approach):C(Support)"]:
    table = result.results[effect]["stat"]
    wilks = table.loc["Wilks' lambda", "Value"]
    f_stat = table.loc["Wilks' lambda", "F Value"]
    p_value = table.loc["Wilks' lambda", "Pr > F"]
    print(effect, "| Wilks Lambda:", round(wilks, 6), "| F:", round(f_stat, 4),
          "| p-value:", p_value, "|", decide(p_value))

C(Approach) | Wilks Lambda: 0.517348 | F: 128.6692 | p-value: 1.1348190231790386e-137 | reject H0
C(Support) | Wilks Lambda: 0.939501 | F: 10.4489 | p-value: 1.9534723008562045e-11 | reject H0
C(Approach):C(Support) | Wilks Lambda: 0.933917 | F: 5.7087 | p-value: 8.416323877365912e-10 | reject H0


### Non-Parametric Tests For Numerical & Ordinal Categorical Inter-Group

#### Wilcoxon Rank-Sum Test For Independent t-Test

In [17]:
group_1 = resample_data.loc[resample_data["Approach"] == "NET", "W10MAND"].values
group_2 = resample_data.loc[resample_data["Approach"] == "DTT", "W10MAND"].values

u_stat, p_value = stats.mannwhitneyu(group_1, group_2, alternative="two-sided")

pooled = np.concatenate([group_1, group_2])
ranks = stats.rankdata(pooled)
observed = ranks[:len(group_1)].sum()
null = np.array([rng.permutation(ranks)[:len(group_1)].sum() for _ in range(n_permutation)])
p_permutation = np.mean(np.abs(null - null.mean()) >= abs(observed - null.mean()))

print("NET:", len(group_1), "| median:", np.median(group_1))
print("DTT:", len(group_2), "| median:", np.median(group_2))
print("U:", u_stat, "| rank sum:", observed)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

NET: 713 | median: 94.0
DTT: 784 | median: 66.0
U: 373405.5 | rank sum: 627946.5
p-value (asymptotic): 2.527182745631487e-29
p-value (permutation): 0.0
Decision: reject H0


#### Wilcoxon Signed-Rank Test For Paired t-Test

In [18]:
differences = (resample_data["W10MAND"] - resample_data["W01MAND"]).values
differences = differences[differences != 0]

w_stat, p_value = stats.wilcoxon(differences)

ranks = stats.rankdata(np.abs(differences))
observed = ranks[differences > 0].sum()
signs = rng.choice([-1, 1], size=(n_permutation, len(ranks)))
null = (ranks * (signs > 0)).sum(axis=1)
p_permutation = np.mean(np.abs(null - null.mean()) >= abs(observed - null.mean()))

print("Pairs:", len(differences), "| median difference:", np.median(differences))
print("W:", w_stat, "| positive rank sum:", observed)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Pairs: 1999 | median difference: 21.0
W: 0.0 | positive rank sum: 1999000.0
p-value (asymptotic): 0.0
p-value (permutation): 0.0
Decision: reject H0


#### Kruskal–Wallis H Test For One-Way ANOVA Test

In [19]:
codes = resample_data["Support"].cat.codes.values
values = resample_data["W10MAND"].values
levels = np.unique(codes)

h_stat, p_value = stats.kruskal(*[values[codes == c] for c in levels])

null = np.empty(n_permutation)
for i in range(n_permutation):
    shuffled = rng.permutation(codes)
    null[i] = stats.kruskal(*[values[shuffled == c] for c in levels]).statistic
p_permutation = np.mean(null >= h_stat)

print(resample_data.groupby("Support", observed=True)["W10MAND"].median())
print("H:", round(h_stat, 4), "| df:", len(levels) - 1)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Support
Level 1    109.0
Level 2     68.0
Level 3     40.0
Name: W10MAND, dtype: float64
H: 1011.2111 | df: 2
p-value (asymptotic): 2.6199835813953245e-220
p-value (permutation): 0.0
Decision: reject H0


#### Scheirer–Ray–Hare (SRH) Test For Two-Way ANOVA Test

In [20]:
srh = resample_data.copy()
srh["Rank"] = stats.rankdata(srh["W10MAND"])
mean_square_total = srh["Rank"].var(ddof=1)

table = anova_lm(ols("Rank ~ C(Support) * C(Approach)", data=srh).fit(), typ=2)
print(table)

for effect in ["C(Support)", "C(Approach)", "C(Support):C(Approach)"]:
    h_stat = table.loc[effect, "sum_sq"] / mean_square_total
    df_effect = int(table.loc[effect, "df"])
    p_value = 1 - stats.chi2.cdf(h_stat, df_effect)
    print(effect, "| H:", round(h_stat, 4), "| df:", df_effect,
          "| p-value:", p_value, "|", decide(p_value))

                              sum_sq      df           F         PR(>F)
C(Support)              2.542395e+08     2.0  791.964935  8.863735e-254
C(Approach)             7.086280e+06     2.0   22.074008   3.297010e-10
C(Support):C(Approach)  2.738540e+06     4.0    4.265323   1.935431e-03
Residual                3.195791e+08  1991.0         NaN            NaN
C(Support) | H: 762.3922 | df: 2 | p-value: 0.0 | reject H0
C(Approach) | H: 21.2497 | df: 2 | p-value: 2.430395222163817e-05 | reject H0
C(Support):C(Approach) | H: 8.2121 | df: 4 | p-value: 0.08411035938309774 | fail to reject H0


#### Friedman Test For One-Way Repeated Measures ANOVA Test

In [21]:
matrix = resample_data[["W01MAND", "W05MAND", "W10MAND"]].values

q_stat, p_value = stats.friedmanchisquare(*matrix.T)

null = np.empty(n_permutation)
for i in range(n_permutation):
    shuffled = rng.permuted(matrix, axis=1)
    null[i] = stats.friedmanchisquare(*shuffled.T).statistic
p_permutation = np.mean(null >= q_stat)

print(pd.DataFrame(matrix, columns=["W01MAND", "W05MAND", "W10MAND"]).median())
print("Q:", round(q_stat, 4), "| df:", matrix.shape[1] - 1)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

W01MAND    49.0
W05MAND    58.0
W10MAND    71.0
dtype: float64
Q: 3981.5173 | df: 2
p-value (asymptotic): 0.0
p-value (permutation): 0.0
Decision: reject H0


#### Aligned Rank Transform Test For Two-Way Repeated Measures ANOVA Test

In [22]:
subjects = resample_data.sample(n=400, random_state=42)
frames = []
for week in ["W01", "W10"]:
    frames.append(pd.DataFrame({
        "ID": subjects["ID"].values,
        "Week": week,
        "Approach": subjects["Approach"].values,
        "Score": subjects[week + "MAND"].values,
    }))
art = pd.concat(frames, ignore_index=True)

grand_mean = art["Score"].mean()
cell_mean = art.groupby(["Week", "Approach"])["Score"].transform("mean")
week_mean = art.groupby("Week")["Score"].transform("mean")
approach_mean = art.groupby("Approach")["Score"].transform("mean")

art["Aligned_Week"] = art["Score"] - cell_mean + (week_mean - grand_mean)
art["Aligned_Approach"] = art["Score"] - cell_mean + (approach_mean - grand_mean)
art["Aligned_Interaction"] = art["Score"] - week_mean - approach_mean + grand_mean

for effect, column in [("C(Week)", "Aligned_Week"),
                       ("C(Approach)", "Aligned_Approach"),
                       ("C(Week):C(Approach)", "Aligned_Interaction")]:
    art["Ranked"] = stats.rankdata(art[column])
    table = anova_lm(ols("Ranked ~ C(Week) * C(Approach)", data=art).fit(), typ=2)
    f_stat = table.loc[effect, "F"]
    p_value = 1 - stats.f.cdf(f_stat, table.loc[effect, "df"], table.loc["Residual", "df"])
    print(effect, "| F:", round(f_stat, 4), "| df:", int(table.loc[effect, "df"]),
          int(table.loc["Residual", "df"]), "| p-value:", p_value, "|", decide(p_value))

C(Week) | F: 91.0497 | df: 1 794 | p-value: 1.1102230246251565e-16 | reject H0
C(Approach) | F: 56.2254 | df: 2 794 | p-value: 1.1102230246251565e-16 | reject H0
C(Week):C(Approach) | F: 1.2496 | df: 2 794 | p-value: 0.287177165384203 | fail to reject H0


#### One-Way PERMANOVA Test For One-Way MANOVA Test

In [23]:
response = distance_data[["W10MAND", "W10TACT", "W10INTRA"]].values.astype(float)
codes = distance_data["Support"].cat.codes.values
n = len(response)

distance = np.linalg.norm(response[:, None, :] - response[None, :, :], axis=2)
squared = distance ** 2
total_sum_of_squares = squared.sum() / (2 * n)

def within_sum_of_squares(labels):
    total = 0.0
    for label in np.unique(labels):
        index = np.where(labels == label)[0]
        total += squared[np.ix_(index, index)].sum() / (2 * len(index))
    return total

def pseudo_f(labels):
    groups = len(np.unique(labels))
    within = within_sum_of_squares(labels)
    return ((total_sum_of_squares - within) / (groups - 1)) / (within / (n - groups))

observed = pseudo_f(codes)
null = np.array([pseudo_f(rng.permutation(codes)) for _ in range(n_permutation)])
p_permutation = (np.sum(null >= observed) + 1) / (n_permutation + 1)

print("Observations:", n, "| Groups:", len(np.unique(codes)))
print("Total sum of squares:", round(total_sum_of_squares, 3))
print("Pseudo-F:", round(observed, 4))
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Observations: 300 | Groups: 3
Total sum of squares: 1401015.72
Pseudo-F: 93.1289
p-value (permutation): 0.000999000999000999
Decision: reject H0


#### Two-Way PERMANOVA Test For Two-Way MANOVA Test

In [24]:
codes_a = distance_data["Support"].cat.codes.values
codes_b = pd.Categorical(distance_data["Approach"]).codes
codes_cell = codes_a * 10 + codes_b

levels_a = len(np.unique(codes_a))
levels_b = len(np.unique(codes_b))
df_a = levels_a - 1
df_b = levels_b - 1
df_ab = df_a * df_b
df_residual = n - levels_a * levels_b

def two_way_pseudo_f(labels_a, labels_b):
    residual = within_sum_of_squares(labels_a * 10 + labels_b)
    ss_a = total_sum_of_squares - within_sum_of_squares(labels_a)
    ss_b = total_sum_of_squares - within_sum_of_squares(labels_b)
    ss_ab = (total_sum_of_squares - residual) - ss_a - ss_b
    mean_residual = residual / df_residual
    return ((ss_a / df_a) / mean_residual,
            (ss_b / df_b) / mean_residual,
            (ss_ab / df_ab) / mean_residual)

observed = two_way_pseudo_f(codes_a, codes_b)
null = np.array([two_way_pseudo_f(rng.permutation(codes_a), rng.permutation(codes_b))
                 for _ in range(n_permutation)])

for i, effect in enumerate(["Support", "Approach", "Support:Approach"]):
    p_permutation = (np.sum(null[:, i] >= observed[i]) + 1) / (n_permutation + 1)
    print(effect, "| Pseudo-F:", round(observed[i], 4),
          "| p-value (permutation):", p_permutation, "|", decide(p_permutation))

Support | Pseudo-F: 94.0685 | p-value (permutation): 0.000999000999000999 | reject H0
Approach | Pseudo-F: 33.1073 | p-value (permutation): 0.000999000999000999 | reject H0
Support:Approach | Pseudo-F: -14.3045 | p-value (permutation): 1.0 | fail to reject H0


#### One-Way Repeated Measures PERMANOVA Test For One-Way Repeated Measures MANOVA Test

In [25]:
weeks = ["W01", "W05", "W10"]
subject_count = len(distance_data)
stacked = np.vstack([distance_data[[w + "MAND", w + "TACT", w + "INTRA"]].values for w in weeks])
stacked = stacked.astype(float)
week_codes = np.repeat(np.arange(len(weeks)), subject_count)
total = len(stacked)

squared_rm = np.linalg.norm(stacked[:, None, :] - stacked[None, :, :], axis=2) ** 2
total_rm = squared_rm.sum() / (2 * total)

def within_rm(labels):
    value = 0.0
    for label in np.unique(labels):
        index = np.where(labels == label)[0]
        value += squared_rm[np.ix_(index, index)].sum() / (2 * len(index))
    return value

def pseudo_f_rm(labels):
    groups = len(np.unique(labels))
    within = within_rm(labels)
    return ((total_rm - within) / (groups - 1)) / (within / (total - groups))

observed = pseudo_f_rm(week_codes)
null = np.empty(n_permutation)
for i in range(n_permutation):
    shuffled = np.argsort(rng.random((len(weeks), subject_count)), axis=0).ravel()
    null[i] = pseudo_f_rm(shuffled)
p_permutation = (np.sum(null >= observed) + 1) / (n_permutation + 1)

print("Subjects:", subject_count, "| Repeated levels:", len(weeks))
print("Pseudo-F:", round(observed, 4))
print("p-value (within-subject permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Subjects: 300 | Repeated levels: 3
Pseudo-F: 31.4111
p-value (within-subject permutation): 0.000999000999000999
Decision: reject H0


#### Two-Way Repeated Measures PERMANOVA Test For Two-Way Repeated Measures MANOVA Test

In [26]:
approach_codes = np.tile(pd.Categorical(distance_data["Approach"]).codes, len(weeks))
levels_week = len(np.unique(week_codes))
levels_approach = len(np.unique(approach_codes))
df_week = levels_week - 1
df_approach = levels_approach - 1
df_interaction = df_week * df_approach
df_error = total - levels_week * levels_approach

def two_way_pseudo_f_rm(labels_week, labels_approach):
    residual = within_rm(labels_week * 10 + labels_approach)
    ss_week = total_rm - within_rm(labels_week)
    ss_approach = total_rm - within_rm(labels_approach)
    ss_interaction = (total_rm - residual) - ss_week - ss_approach
    mean_residual = residual / df_error
    return ((ss_week / df_week) / mean_residual,
            (ss_approach / df_approach) / mean_residual,
            (ss_interaction / df_interaction) / mean_residual)

observed = two_way_pseudo_f_rm(week_codes, approach_codes)
null = np.empty((n_permutation, 3))
for i in range(n_permutation):
    shuffled_week = np.argsort(rng.random((levels_week, subject_count)), axis=0).ravel()
    shuffled_subject = np.tile(rng.permutation(approach_codes[:subject_count]), levels_week)
    null[i] = two_way_pseudo_f_rm(shuffled_week, shuffled_subject)

for i, effect in enumerate(["Week", "Approach", "Week:Approach"]):
    p_permutation = (np.sum(null[:, i] >= observed[i]) + 1) / (n_permutation + 1)
    print(effect, "| Pseudo-F:", round(observed[i], 4),
          "| p-value (permutation):", p_permutation, "|", decide(p_permutation))

Week | Pseudo-F: 35.6259 | p-value (permutation): 0.000999000999000999 | reject H0
Approach | Pseudo-F: 61.3061 | p-value (permutation): 0.000999000999000999 | reject H0
Week:Approach | Pseudo-F: 0.9373 | p-value (permutation): 0.000999000999000999 | reject H0


### Non-Parametric Test For Nominal Categorical Inter-Group

#### Cramer’s V With Chi-Square Homogeneity Test

In [27]:
gain = data["W10MAND"] - data["W01MAND"]
data["Response"] = np.where(gain > gain.median(), "High Gain", "Low Gain")

table = pd.crosstab(data["Approach"], data["Response"])
chi_square, p_value, degrees, expected = stats.chi2_contingency(table)
v = np.sqrt(chi_square / (table.values.sum() * (min(table.shape) - 1)))

print(table)
print("Chi-square:", round(chi_square, 4), "| df:", degrees)
print("Cramer's V:", round(v, 4))
print("Relationship strength:", interpret(v, [0.1, 0.3, 0.5],
                                          ["negligible", "weak", "moderate", "strong"]))
print("p-value (upper-tailed):", p_value)
print("Decision:", decide(p_value))

Response  High Gain  Low Gain
Approach                     
DTT            2363      5029
NET            4396      2351
PECS           2386      2825
Chi-square: 1564.9674 | df: 2
Cramer's V: 0.2844
Relationship strength: weak
p-value (upper-tailed): 0.0
Decision: reject H0


## INTER-VARIABLE ANALYSIS

### Parametric Test For Continuous Numerical Inter-Variables

#### Pearson Correlation Test

In [28]:
x = data["MAND0"]
y = data["INTRA0"]

r, p_value = stats.pearsonr(x, y)
n = len(x)
t_stat = r * np.sqrt((n - 2) / (1 - r ** 2))
p_manual = 2 * (1 - stats.t.cdf(abs(t_stat), n - 2))

print("Pairs:", n)
print("Pearson r:", round(r, 4))
print("Direction:", "positive" if r > 0 else "negative")
print("Relationship:", interpret(r, [0.3, 0.7, 1.0],
                                 ["weak", "moderate", "strong", "perfect"]))
print("t:", round(t_stat, 4), "| df:", n - 2)
print("p-value (two-tailed):", p_value)
print("p-value (CDF):", p_manual)
print("Decision:", decide(p_value))

Pairs: 19350
Pearson r: 0.9272
Direction: positive
Relationship: strong
t: 344.2624 | df: 19348
p-value (two-tailed): 0.0
p-value (CDF): 0.0
Decision: reject H0


### Non-Parametric Test For Numerical & Ordinal Categorical Inter-Variables

#### Spearman Rank Correlation Test

In [29]:
x = resample_data["MAND0"].values
y = resample_data["Support"].cat.codes.values

rho, p_value = stats.spearmanr(x, y)

null = np.array([stats.spearmanr(x, rng.permutation(y)).statistic
                 for _ in range(n_permutation)])
p_permutation = (np.sum(np.abs(null) >= abs(rho)) + 1) / (n_permutation + 1)

print("Pairs:", len(x))
print("Spearman rho:", round(rho, 4))
print("Direction:", "positive" if rho > 0 else "negative")
print("Relationship:", interpret(rho, [0.3, 0.7, 1.0],
                                 ["weak", "moderate", "strong", "perfect"]))
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Pairs: 2000
Spearman rho: -0.7181
Direction: negative
Relationship: strong
p-value (asymptotic): 6.954641e-317
p-value (permutation): 0.000999000999000999
Decision: reject H0


### Non-Parametric Test For Nominal Categorical Inter-Variables

#### Cramer’s V With Chi-Square Independence Test

In [30]:
table = pd.crosstab(data["Approach"], data["Support"])
chi_square, p_value, degrees, expected = stats.chi2_contingency(table)
v = np.sqrt(chi_square / (table.values.sum() * (min(table.shape) - 1)))

print(table)
print("Chi-square:", round(chi_square, 4), "| df:", degrees)
print("Cramer's V:", round(v, 4))
print("Relationship strength:", interpret(v, [0.1, 0.3, 0.5],
                                          ["negligible", "weak", "moderate", "strong"]))
print("p-value (upper-tailed):", p_value)
print("Decision:", decide(p_value))

Support   Level 1  Level 2  Level 3
Approach                           
DTT          2530     3008     1854
NET          3807     2203      737
PECS          725     1442     3044
Chi-square: 3976.7942 | df: 4
Cramer's V: 0.3206
Relationship strength: moderate
p-value (upper-tailed): 0.0
Decision: reject H0
